# Step 10: Figures and tables for the manuscript

Reads the tidy result files of `step09_cox_models.ipynb` (step 9) (`RESULTS_DIR`) and the diagnostics of
`step08_cox_dataprep.ipynb` (step 8) (`DATAPREP_DIR`) and writes all figures/tables to `FIG_DIR`.

**HSR requirements:** print area 10.7 cm × 17.1 cm, figures must work in black and white,
figures as TIFF with at least 300 dpi. Every figure is therefore saved as PDF (for LaTeX) **and** TIFF (300 dpi, for submission),
at a width of 10.7 cm, using line styles / marker shapes instead of colours to distinguish groups.

| output | content | suggested place |
|---|---|---|
| `fig_classification_artifact` | share of F/M-classified authors and female share among them, by entry year and career span | Methods / data section |
| `fig_gender_hr_models` | female/male hazard ratio across Models 0–5 (main spec) | Results |
| `fig_gender_hr_specs` | female/male hazard ratio across models and specifications | Results |
| `fig_gender_hr_cohorts` | female/male hazard ratio by entry period | Results |
| `fig_M3_effects` | effects of entry conditions for men and women (Model 3, main spec) | Results |
| `fig_linear_effects` | linear effects over the full range (Model 2 all authors; Model 3 men/women) | Results |
| `fig_splines` | non-linear effects over the full range (Model 4 all authors; Model 5 men/women) | Results |
| `fig_splines_gap` | gender gap (women vs. men HR) over the full range (Model 5 splines; Model 3 linear) | Results |
| `tab_main_models.tex/.csv` | Models 0–5, main specification | Results |
| `tab_sample_flow.tex/.csv` | sample construction | Methods |

## 0. Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

BASE_DIR = os.path.abspath('..')                  # root of the replication package (notebooks run from code/)
RESULTS_DIR = f'{BASE_DIR}/results'                # output of step 9
DATAPREP_DIR = f'{BASE_DIR}/data/derived'          # output of step 8
FIG_DIR = f'{BASE_DIR}/figures/models'
MAIN = 'main'            # main specification (entry 1950–2018): M3 effects, splines and the model table
os.makedirs(FIG_DIR, exist_ok=True)

CM = 1 / 2.54
WIDTH = 10.7 * CM          # HSR print width
mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'STIX Two Text', 'DejaVu Serif'],
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.6, 'lines.linewidth': 1.0,
    'savefig.dpi': 300,
})
# black-and-white friendly styles
STYLE = {
    'men':     dict(color='black',   linestyle='--',  marker='o'),
    'women':   dict(color='0.45',    linestyle='-', marker='^'),
    'overall': dict(color='black',   linestyle='-',  marker='s'),
    'gap':     dict(color='black',   linestyle='-',  marker='D'),
}

def save(fig, name):
    fig.savefig(f'{FIG_DIR}/{name}.pdf', bbox_inches='tight')
    fig.savefig(f'{FIG_DIR}/{name}.tiff', dpi=300, bbox_inches='tight', pil_kwargs={'compression': 'tiff_lzw'})
    print('saved', name)

## 1. Load results

In [ ]:
import glob
def read_all(name):
    """Concatenate one result file over all specification folders (RESULTS_DIR/<spec>/<name>.csv)."""
    files = sorted(glob.glob(f'{RESULTS_DIR}/*/{name}.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else None

coef = read_all('coef')
mstats = read_all('stats')
vcov = read_all('vcov')
profiles = read_all('profiles')
splines = read_all('splines')
flow = pd.read_csv(f'{DATAPREP_DIR}/sample_flow.csv')
diag = pd.read_csv(f'{DATAPREP_DIR}/diag_classification_by_entry_year_span.csv')

def V(spec, model):
    """Covariance matrix of one model as DataFrame."""
    v = vcov[(vcov.spec == spec) & (vcov.model == model)]
    return v.pivot(index='term_1', columns='term_2', values='cov')

def combo(spec, model, weights):
    """HR and 95% CI of sum_k w_k beta_k."""
    c = coef[(coef.spec == spec) & (coef.model == model)].set_index('term')['coef']
    names = list(weights)
    w = np.array([weights[n] for n in names])
    est = float(w @ c.loc[names].values)
    se = float(np.sqrt(w @ V(spec, model).loc[names, names].values @ w))
    return np.exp(est), np.exp(est - 1.959964 * se), np.exp(est + 1.959964 * se)

## 2. Data-quality figure: gender classification depends on career length before 2002

In [ ]:
span_order = ['0', '1-4', '5-9', '10+']
span_label = {'0': 'active 1 year', '1-4': '2–5 years', '5-9': '6–10 years', '10+': '> 10 years'}
span_style = {'0': ('-', 'black'), '1-4': ('--', 'black'), '5-9': (':', 'black'), '10+': ('-.', '0.5')}
dd = diag[(diag.begin_year >= 1950) & (diag.begin_year <= 2018)]   # full observation window

fig, axes = plt.subplots(2, 1, figsize=(WIDTH, 11 * CM), sharex=True)
for s in span_order:
    sub = dd[dd.career_span.astype(str) == s].sort_values('begin_year')
    ls, col = span_style[s]
    axes[0].plot(sub.begin_year, sub.share_classified_fm * 100, linestyle=ls, color=col, label=span_label[s])
    axes[1].plot(sub.begin_year, sub.female_share_among_classified * 100, linestyle=ls, color=col, label=span_label[s])
for ax in axes:
    ax.axvline(2002, color='0.6', linewidth=0.6)
    ax.grid(axis='y', linewidth=0.3, alpha=0.5)
axes[0].set_ylabel('classified as F or M (%)')
axes[1].set_ylabel('women among classified (%)')
axes[1].set_xlabel('entry year')
axes[1].legend(title='career span', frameon=False, loc='upper left')
axes[0].annotate('2002: NLM starts recording\nfull first names', xy=(2002, 60), xytext=(2004, 55), fontsize=6, color='0.3')
fig.tight_layout()
save(fig, 'fig_classification_artifact')

## 3. Gender hazard ratio across models and specifications

In [ ]:
spec_label = {
    'main': 'Main: entry 1950–2018,\n≥2 active years',
    'check_2002': 'entry 2002–2018',
    'check_pre2002': 'entry 1950–2001',
    'check_gap3': 'dropout = 3-year gap',
    'check_all_authors': 'incl. one-year authors',
    'check_pkg_2002': 'entry 2002–2018,\nonly gender in PKG',
}
model_label = {'M0': 'M0 raw', 'M1': 'M1 + cohort strata', 'M2': 'M2 + entry conditions',
               'M3': 'M3 interactions (at median)', 'M2_field': 'M2 + field strata'}
g = coef[(coef.term == 'gender_f') & coef.model.isin(model_label)]
specs = [s for s in spec_label if s in g.spec.unique()]
models = [m for m in model_label if m in g.model.unique()]

fig, ax = plt.subplots(figsize=(WIDTH, 10.5 * CM))
markers = ['o', 's', '^', 'D', 'v']
offsets = np.linspace(-0.3, 0.3, len(models))
for k, m in enumerate(models):
    sub = g[g.model == m].set_index('spec').reindex(specs)
    y = np.arange(len(specs)) + offsets[k]
    ax.errorbar(sub.HR, y, xerr=[sub.HR - sub.HR_low, sub.HR_high - sub.HR], fmt=markers[k], ms=3.5,
                color='black' if k % 2 == 0 else '0.5', mfc='white' if k >= 2 else None,
                capsize=1.5, elinewidth=0.6, label=model_label[m])
ax.axvline(1, color='0.5', linewidth=0.6, linestyle=':')
ax.set_yticks(range(len(specs)))
ax.set_yticklabels([spec_label[s] for s in specs])
ax.invert_yaxis()
ax.set_xlabel('hazard ratio of dropout, women vs. men')
ax.legend(frameon=False, fontsize=6, ncol=2, loc='upper center', bbox_to_anchor=(0.35, -0.14))
fig.tight_layout()
save(fig, 'fig_gender_hr_specs')

## 3b. Gender hazard ratio across models (main spec)

Models on the x-axis, HR of `gender_f` on the y-axis.
In Models 3 and 5 (gender interactions, entry conditions centred at the median) `gender_f` is the gender gap at median
entry conditions with one first-year paper; these are shown with hollow markers.

In [ ]:
from matplotlib.lines import Line2D

# Figure: hazard ratio of gender_f (women vs. men) across Models 0-5, main specification
model_names = {'M0': 'Model 0', 'M1': 'Model 1', 'M2': 'Model 2', 'M3': 'Model 3', 'M4': 'Model 4', 'M5': 'Model 5'}
INTERACTION_MODELS = ['M3', 'M5']   # gender_f = gender gap at the median of all entry conditions
gm = (coef[(coef.spec == MAIN) & (coef.term == 'gender_f') & coef.model.isin(model_names)]
      .set_index('model').reindex([m for m in model_names if m in set(coef.model)]).dropna(subset=['HR']))

fig, ax = plt.subplots(figsize=(WIDTH, 6 * CM))
x = np.arange(len(gm))
for xi, (m, r) in zip(x, gm.iterrows()):
    hollow = m in INTERACTION_MODELS
    ax.errorbar(xi, r.HR, yerr=[[r.HR - r.HR_low], [r.HR_high - r.HR]], fmt='o', ms=4, color='black',
                mfc='white' if hollow else 'black', capsize=2, elinewidth=0.6)
ax.axhline(1, color='0.5', linewidth=0.6, linestyle=':')
ax.set_xticks(x)
ax.set_xticklabels([model_names[m] for m in gm.index])
ax.set_ylabel('hazard ratio of dropout,\nwomen vs. men')
ax.grid(axis='y', linewidth=0.3, alpha=0.5)
ax.legend(handles=[Line2D([0], [0], marker='o', color='black', linestyle='none', ms=4, label='without gender interactions'),
                   Line2D([0], [0], marker='o', color='black', mfc='white', linestyle='none', ms=4,
                          label='with gender interactions (gap at median entry conditions)')],
          frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=1)
fig.tight_layout()
save(fig, 'fig_gender_hr_models')
gm[['HR', 'HR_low', 'HR_high']].round(3)

## 4. Gender hazard ratio by entry period (Model 2 with period-specific gender effects)

In [ ]:
cc = coef[(coef.model == 'M2_cohort') & coef.term.str.startswith('f_period_')].copy()
cc['period'] = cc.term.str.replace('f_period_', '', regex=False)
show = [s for s in ['main'] if s in cc.spec.unique()]
fig, axes = plt.subplots(1, len(show), figsize=(WIDTH, 5.5 * CM), sharey=True, squeeze=False)
for ax, s in zip(axes[0], show):
    sub = cc[cc.spec == s]
    x = np.arange(len(sub))
    ax.errorbar(x, sub.HR, yerr=[sub.HR - sub.HR_low, sub.HR_high - sub.HR], fmt='o', color='black',
                ms=3, capsize=1.5, elinewidth=0.6)
    ax.axhline(1, color='0.5', linewidth=0.6, linestyle=':')
    ax.set_xticks(x)
    ax.set_xticklabels(sub.period, rotation=45, ha='right')
    #ax.set_title({'main': 'Gendered HR by entry period'}[s])
axes[0][0].set_ylabel('HR women vs. men')
fig.tight_layout()
save(fig, 'fig_gender_hr_cohorts')

## 5. Model 3 (main spec): effect of each entry condition for men and women

Entry conditions on the x-axis, HR on the y-axis.
Men: main effect. Women: main effect + interaction (CI from the covariance matrix).
Units: female share per 10 percentage points, topic size per log unit (≈ factor 2.7), maturity per 5 years,
first-year papers vs. one paper. Men = black circles, women = grey triangles (works in black and white).

In [ ]:
terms = [('fs10', 1, 'female share\n(+10 pp)'), ('size_log', 1, 'topic size\n(log, +1)'),
         ('maturity', 5, 'topic maturity\n(+5 years)'), ('fyp_2', 1, '2 first-year\npapers'),
         ('fyp_3plus', 1, '3+ first-year\npapers')]
rows = []
for t, scale, lab in terms:
    men = combo(MAIN, 'M3', {t: scale})
    women = combo(MAIN, 'M3', {t: scale, f'f_x_{t}': scale})
    rows += [dict(term=t, label=lab.replace('\n', ' '), group='men', HR=men[0], lo=men[1], hi=men[2]),
             dict(term=t, label=lab.replace('\n', ' '), group='women', HR=women[0], lo=women[1], hi=women[2])]
eff = pd.DataFrame(rows)
eff.to_csv(f'{FIG_DIR}/tab_M3_effects_by_gender.csv', index=False)

fig, ax = plt.subplots(figsize=(WIDTH, 7 * CM))
x = np.arange(len(terms))
for grp, off in [('men', -0.12), ('women', 0.12)]:
    sub = eff[eff.group == grp]
    st = STYLE[grp]
    ax.errorbar(x + off, sub.HR, yerr=[sub.HR - sub.lo, sub.hi - sub.HR], fmt=st['marker'], color=st['color'],
                ms=4, capsize=2, elinewidth=0.6, label=grp)
ax.axhline(1, color='0.5', linewidth=0.6, linestyle=':')
ax.set_xticks(x)
ax.set_xticklabels([t[2] for t in terms])
ax.set_ylabel('hazard ratio of dropout')
ax.grid(axis='y', linewidth=0.3, alpha=0.5)
ax.legend(frameon=False, loc='lower left')
fig.tight_layout()
save(fig, 'fig_M3_effects')

hr, lo, hi = combo(MAIN, 'M3', {'gender_f': 1})
print(f'gender_f (women vs. men at median entry conditions, one first-year paper): HR={hr:.3f} [{lo:.3f}, {hi:.3f}]')
eff.round(3)

## 6. Linear effects over the full range of each entry condition (Models 2 and 3, main spec)

Hazard ratio relative to the sample median of the variable,
evaluated over **all observed values** (minimum to maximum of the main analysis sample, not only 1st–99th percentile).
Model 2 = all authors (no gender interaction), Model 3 = men (main effect) and women (main + interaction).
Black-and-white coding: dotted grey = all authors, solid black = women, dashed black = men; 95% CI as grey band (women, all) or hatching (men).

The observed ranges are read once from `cox_data.csv` (same sample definition as in `step09_cox_models.ipynb`) and cached in `FIG_DIR/variable_ranges_main.csv`.

In [ ]:
import json
# Observed range (min..max) of each entry variable in the main analysis sample -> curves cover all values.
# Computed once from the analysis data (same sample definition as in step09_cox_models) and cached.
meta = json.load(open(f'{RESULTS_DIR}/{MAIN}/sample_meta.json'))[0]
REF = meta['reference_values']                                       # medians: fs10, size_log, maturity
KNOTS = meta['knots']                                                # spline knots: fs, size, maturity
RANGE_FILE = f'{FIG_DIR}/variable_ranges_{MAIN}.csv'
if os.path.exists(RANGE_FILE):
    rng = pd.read_csv(RANGE_FILE, index_col=0)
else:
    lo_y, hi_y = meta['entry']
    d = pd.read_csv(f'{DATAPREP_DIR}/cox_data.csv',
                    usecols=['begin_year', 'one_year', 'duration_2y', 'topic_size_entry',
                             'topic_maturity_entry', 'female_share_entry'])
    d = d[(d.begin_year >= lo_y) & (d.begin_year <= hi_y) & (d.duration_2y > 0)]
    if meta['o1y']:
        d = d[d.one_year == 0]
    assert len(d) == meta['n'], 'sample does not match the model sample'
    raw = pd.DataFrame({'fs10': d.female_share_entry * 10,
                        'size_log': np.log1p(d.topic_size_entry),
                        'maturity': d.topic_maturity_entry.astype(float)})
    rng = raw.agg(['min', 'max']).T
    rng.to_csv(RANGE_FILE)
    del d, raw
rng

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# display scale of each variable (model scale -> axis)
VARS = {  # linear term, spline stem, axis label, transform, log x-axis?
    'fs10':     ('fs',       'female share at entry (%)',          lambda x: x * 10, False),
    'size_log': ('size',     'topic size at entry (papers, log scale)', np.expm1,    True),
    'maturity': ('maturity', 'topic maturity at entry (years)',    lambda x: x,      False),
}
# black-and-white curve styles: line style + CI band (grey fill vs. hatching)
CURVE = {
    'overall': dict(line=dict(color='0.35', linestyle=':', linewidth=1.3),
                    band=dict(color='0.35', alpha=0.15, linewidth=0)),
    'men':     dict(line=dict(color='black', linestyle='--'),
                    band=dict(facecolor='none', edgecolor='0.45', hatch='/////', linewidth=0)),
    'women':   dict(line=dict(color='black', linestyle='-'),
                    band=dict(color='black', alpha=0.20, linewidth=0)),
}
LABEL = {'overall': 'all authors (no interaction)', 'men': 'men', 'women': 'women'}

def legend_handles(kinds):
    return [(Line2D([0], [0], **CURVE[k]['line']), Patch(**CURVE[k]['band'])) for k in kinds], [LABEL[k] for k in kinds]

def lin_curve(spec, model, weights, dx):
    """HR (and 95% CI) of dx * sum_k w_k beta_k along a grid of distances dx to the reference value."""
    c = coef[(coef.spec == spec) & (coef.model == model)].set_index('term')['coef']
    names = list(weights)
    w = np.array([weights[n] for n in names], dtype=float)
    b = float(w @ c.loc[names].values)
    se = float(np.sqrt(w @ V(spec, model).loc[names, names].values @ w))
    est = b * dx
    half = 1.959964 * se * np.abs(dx)
    return np.exp(est), np.exp(est - half), np.exp(est + half)

def draw(ax, xd, hr, lo, hi, kind):
    ax.fill_between(xd, lo, hi, **CURVE[kind]['band'])
    ax.plot(xd, hr, **CURVE[kind]['line'])

def style_panel(ax, label, logx):
    ax.axhline(1, color='0.5', linewidth=0.6, linestyle=':')
    ax.set_xlabel(label)
    ax.set_ylabel('hazard ratio vs. median')
    ax.grid(linewidth=0.3, alpha=0.5)
    if logx:
        ax.set_xscale('log')

# Linear effects over the full observed range: Model 2 (all authors) and Model 3 (men / women)
fig, axes = plt.subplots(len(VARS), 1, figsize=(WIDTH, 16 * CM))
for ax, (v, (stem, lab, tr, logx)) in zip(axes, VARS.items()):
    grid = np.linspace(rng.loc[v, 'min'], rng.loc[v, 'max'], 200)
    dx = grid - REF[v]
    draw(ax, tr(grid), *lin_curve(MAIN, 'M2', {v: 1}, dx), 'overall')
    draw(ax, tr(grid), *lin_curve(MAIN, 'M3', {v: 1}, dx), 'men')
    draw(ax, tr(grid), *lin_curve(MAIN, 'M3', {v: 1, f'f_x_{v}': 1}, dx), 'women')
    style_panel(ax, lab, logx)
h, l = legend_handles(['overall', 'men', 'women'])
fig.legend(h, l, frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, 1.0), handlelength=3)
fig.tight_layout(rect=[0, 0, 1, 0.965])
save(fig, 'fig_linear_effects')

## 7. Non-linear effects (Models 4 and 5, main spec)

Restricted cubic splines, hazard ratio relative to the median,
over **all observed values**. The curves are recomputed from the coefficients, the covariance matrix and the knots
(`sample_meta.json`), so they are not limited to the 1st–99th percentile grid of `splines.csv`; outside the outer knots
the splines are linear by construction. Small ticks at the bottom of each panel mark the knot positions.
Model 4 = all authors, Model 5 = men and women. Same black-and-white coding as section 6.

In [ ]:
def rcs_basis(x, knots):
    """Restricted cubic spline basis, identical to step09_cox_models (Harrell 2015)."""
    x = np.asarray(x, dtype=float)
    k = np.asarray(knots, dtype=float)
    K = len(k)
    cube = lambda u: np.maximum(u, 0.0) ** 3
    cols = [x]
    for j in range(K - 2):
        term = (cube(x - k[j])
                - cube(x - k[K - 2]) * (k[K - 1] - k[j]) / (k[K - 1] - k[K - 2])
                + cube(x - k[K - 1]) * (k[K - 2] - k[j]) / (k[K - 1] - k[K - 2]))
        cols.append(term / (k[K - 1] - k[0]) ** 2)
    return np.column_stack(cols)

def spline_curve(spec, model, stem, ref, grid, women=False):
    """HR (95% CI) relative to the median from a spline model; women = main + interaction terms."""
    B = rcs_basis(grid, KNOTS[stem]) - rcs_basis([ref], KNOTS[stem])
    cols = [f'{stem}_rcs{j}' for j in range(B.shape[1])]
    if women:
        cols = cols + [f'f_x_{c}' for c in cols]
        B = np.hstack([B, B])
    c = coef[(coef.spec == spec) & (coef.model == model)].set_index('term')['coef'].loc[cols].values
    Vm = V(spec, model).loc[cols, cols].values
    est = B @ c
    se = np.sqrt(np.einsum('ij,jk,ik->i', B, Vm, B))
    return np.exp(est), np.exp(est - 1.959964 * se), np.exp(est + 1.959964 * se)

# Non-linear effects over the full observed range: Model 4 (all authors) and Model 5 (men / women)
fig, axes = plt.subplots(len(VARS), 1, figsize=(WIDTH, 16 * CM))
for ax, (v, (stem, lab, tr, logx)) in zip(axes, VARS.items()):
    grid = np.linspace(rng.loc[v, 'min'], rng.loc[v, 'max'], 200)
    draw(ax, tr(grid), *spline_curve(MAIN, 'M4', stem, REF[v], grid), 'overall')
    draw(ax, tr(grid), *spline_curve(MAIN, 'M5', stem, REF[v], grid), 'men')
    draw(ax, tr(grid), *spline_curve(MAIN, 'M5', stem, REF[v], grid, women=True), 'women')
    for k in KNOTS[stem]:                                # knot positions as small ticks
        ax.axvline(tr(k), ymax=0.03, color='0.4', linewidth=0.6)
    style_panel(ax, lab, logx)
h, l = legend_handles(['overall', 'men', 'women'])
fig.legend(h, l, frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, 1.0), handlelength=3)
fig.tight_layout(rect=[0, 0, 1, 0.965])
save(fig, 'fig_splines')

# consistency check against the spline curves exported by step09_cox_models (1st-99th percentile grid)
if splines is not None:
    for v, (stem, *_ ) in VARS.items():
        for kind, model, wom in [('overall', 'M4', False), ('men', 'M5', False), ('women', 'M5', True)]:
            s = splines[(splines.spec == MAIN) & (splines.model == model) & (splines.variable == stem) & (splines.kind == kind)]
            hr = spline_curve(MAIN, model, stem, REF[v], s.x_raw.values, women=wom)[0]
            assert np.allclose(hr, s.HR.values, rtol=1e-6), (stem, kind)
    print('spline curves match splines.csv')

## 7b. Gender gap over the range of each entry condition (Models 3 and 5, main spec)

Hazard ratio **women vs. men** at each value of an entry condition (all other entry conditions at the median,
one first-year paper). This is the direct comparison that `fig_splines` does not show (there, each curve is relative
to the same gender at the median).
Model 5 (splines): gap(x) = exp(β_female + Σ_j γ_j · [B_j(x) − B_j(median)]), with γ_j the female × spline interactions;
Model 3 (linear, for comparison): gap(x) = exp(β_female + γ · (x − median)).
95% CIs from the full covariance matrix (pointwise). At the median both curves equal the gender HR of the respective model.
Black-and-white coding: solid black + grey band = Model 5 (splines), dotted grey + light band = Model 3 (linear).
Knot positions as small ticks at the bottom; beyond the outer knots (5th/95th percentile) the spline curve is linear by
construction and rests on few observations.
Also writes `tab_gender_gap_splines.csv` (gap at the knots, the median and the observed min/max) for the text.

In [ ]:
def gap_curve_spline(spec, model, stem, ref, grid):
    """Women vs. men HR (95% CI) along grid from a spline interaction model (other conditions at the median)."""
    B = rcs_basis(grid, KNOTS[stem]) - rcs_basis([ref], KNOTS[stem])
    cols = ['gender_f'] + [f'f_x_{stem}_rcs{j}' for j in range(B.shape[1])]
    W = np.column_stack([np.ones(len(grid)), B])
    c = coef[(coef.spec == spec) & (coef.model == model)].set_index('term')['coef'].loc[cols].values
    Vm = V(spec, model).loc[cols, cols].values
    est = W @ c
    se = np.sqrt(np.einsum('ij,jk,ik->i', W, Vm, W))
    return np.exp(est), np.exp(est - 1.959964 * se), np.exp(est + 1.959964 * se)

def gap_curve_linear(spec, model, v, ref, grid):
    """Women vs. men HR (95% CI) along grid from the linear interaction model."""
    cols = ['gender_f', f'f_x_{v}']
    W = np.column_stack([np.ones(len(grid)), np.asarray(grid) - ref])
    c = coef[(coef.spec == spec) & (coef.model == model)].set_index('term')['coef'].loc[cols].values
    Vm = V(spec, model).loc[cols, cols].values
    est = W @ c
    se = np.sqrt(np.einsum('ij,jk,ik->i', W, Vm, W))
    return np.exp(est), np.exp(est - 1.959964 * se), np.exp(est + 1.959964 * se)

GAP_CURVE = {
    'M5': dict(line=dict(color='black', linestyle='-'), band=dict(color='black', alpha=0.20, linewidth=0)),
    'M3': dict(line=dict(color='0.35', linestyle=':', linewidth=1.3), band=dict(color='0.35', alpha=0.12, linewidth=0)),
}
GAP_LABEL = {'M5': 'splines (Model 5)', 'M3': 'linear (Model 3)'}

fig, axes = plt.subplots(len(VARS), 1, figsize=(WIDTH, 16 * CM))
gap_rows = []
for ax, (v, (stem, lab, tr, logx)) in zip(axes, VARS.items()):
    grid = np.linspace(rng.loc[v, 'min'], rng.loc[v, 'max'], 200)
    for m, fn in [('M3', lambda g: gap_curve_linear(MAIN, 'M3', v, REF[v], g)),
                  ('M5', lambda g: gap_curve_spline(MAIN, 'M5', stem, REF[v], g))]:
        hr, lo, hi = fn(grid)
        ax.fill_between(tr(grid), lo, hi, **GAP_CURVE[m]['band'])
        ax.plot(tr(grid), hr, **GAP_CURVE[m]['line'])
        # table for the text: observed min, outer/inner knots, median, observed max
        pts = {'min': rng.loc[v, 'min'], 'median': REF[v], 'max': rng.loc[v, 'max'],
               **{f'knot{j + 1}': k for j, k in enumerate(KNOTS[stem])}}
        hr_p, lo_p, hi_p = fn(np.array(list(pts.values())))
        for (name, x), h, l, u in zip(pts.items(), hr_p, lo_p, hi_p):
            gap_rows.append(dict(variable=v, model=m, point=name, x_display=float(tr(np.array([x]))[0]),
                                 HR=h, HR_low=l, HR_high=u))
    for k in KNOTS[stem]:
        ax.axvline(tr(k), ymax=0.03, color='0.4', linewidth=0.6)
    ax.axhline(1, color='0.5', linewidth=0.6, linestyle=':')
    ax.set_xlabel(lab)
    ax.set_ylabel('HR women vs. men')
    ax.grid(linewidth=0.3, alpha=0.5)
    if logx:
        ax.set_xscale('log')
h = [(Line2D([0], [0], **GAP_CURVE[m]['line']), Patch(**GAP_CURVE[m]['band'])) for m in ['M5', 'M3']]
fig.legend(h, [GAP_LABEL[m] for m in ['M5', 'M3']], frameon=False, ncol=2, loc='upper center',
           bbox_to_anchor=(0.5, 1.0), handlelength=3)
fig.tight_layout(rect=[0, 0, 1, 0.965])
save(fig, 'fig_splines_gap')

gap_tab = pd.DataFrame(gap_rows)
gap_tab.to_csv(f'{FIG_DIR}/tab_gender_gap_splines.csv', index=False)
gap_tab.round(3)

## 8. Tables


In [ ]:
def fmt(r):
    return f"{r.HR:.3f} [{r.HR_low:.3f}, {r.HR_high:.3f}]"

main_models = [m for m in ['M0', 'M1', 'M2', 'M3', 'M4', 'M5'] if m in coef[coef.spec == MAIN].model.unique()]
term_label = {'gender_f': 'Female', 'fs10': 'Female share (+10 pp)', 'size_log': 'Topic size (log)',
              'maturity': 'Topic maturity (years)', 'fyp_2': 'First-year papers: 2', 'fyp_3plus': 'First-year papers: 3+',
              'f_x_fs10': 'Female × female share', 'f_x_size_log': 'Female × topic size',
              'f_x_maturity': 'Female × maturity', 'f_x_fyp_2': 'Female × 2 papers', 'f_x_fyp_3plus': 'Female × 3+ papers'}
c = coef[(coef.spec == MAIN) & coef.model.isin(main_models) & coef.term.isin(term_label)].copy()
c['cell'] = c.apply(fmt, axis=1)
tab = c.pivot(index='term', columns='model', values='cell').reindex(list(term_label)).dropna(how='all')
tab.index = [term_label[t] for t in tab.index]
st_ = mstats[(mstats.spec == MAIN) & mstats.model.isin(main_models)].set_index('model')
for lab, col, f in [('Authors', 'n', '{:,.0f}'), ('Dropouts', 'events', '{:,.0f}'),
                    ('Partial log-likelihood', 'log_likelihood', '{:,.0f}'), ('Concordance', 'concordance', '{:.3f}')]:
    tab.loc[lab] = [f.format(st_.loc[m, col]) if pd.notna(st_.loc[m, col]) else '' for m in tab.columns]
tab.loc['Entry-year strata'] = ['no' if m == 'M0' else 'yes' for m in tab.columns]
tab.loc['Splines (fs, size, maturity)'] = ['yes' if m in ('M4', 'M5') else 'no' for m in tab.columns]
tab = tab.fillna('')
tab.to_csv(f'{FIG_DIR}/tab_main_models.csv')
with open(f'{FIG_DIR}/tab_main_models.tex', 'w') as f:
    f.write(tab.to_latex(escape=True, column_format='l' + 'c' * tab.shape[1]))
tab

In [ ]:
flow.to_csv(f'{FIG_DIR}/tab_sample_flow.csv', index=False)
with open(f'{FIG_DIR}/tab_sample_flow.tex', 'w') as f:
    f.write(flow.to_latex(index=False, escape=True, formatters={c: '{:,.0f}'.format for c in flow.columns if c != 'step'}))
flow